# Stage 2 — AI-Orchestrated Retention Campaigns

**Inputs (from Stage 1)**
- `../outputs/stage1/high_risk_customers.csv` — individual customers with churn probabilities, segments, and top drivers.

**Stage 2 output**
- `customer_campaign_plans.json` — per-customer strategy (intervention_type, channel, offer_type) and message (email_subject, email_body, sms_body).

## 1. Setup and agent infrastructure

Configures paths (`outputs/stage1`, `outputs/stage2`), the Ollama client (`call_ollama`), and run IDs. Defines `AgentIO`, `AgentConfig`, and `OllamaAgent` — a JSON-in/JSON-out base class used by StrategyAgent and ContentAgent.


In [8]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests

# Paths
DATA_PATH = Path("../outputs")
STAGE1_DIR = DATA_PATH / "stage1"
STAGE2_DIR = DATA_PATH / "stage2"
STAGE2_DIR.mkdir(parents=True, exist_ok=True)

HIGH_RISK_PATH = STAGE1_DIR / "high_risk_customers.csv"

# Ollama configuration
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2:3b"


def call_ollama(prompt: str, *, temperature: float = 0.3, max_tokens: int = 512) -> str:
    """Call a local Ollama model and return the generated text.

    This function uses the streaming API but buffers the content into a single string
    for simplicity inside the notebook.
    """
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": True,
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens,
        },
    }

    response = requests.post(url, json=payload, stream=True)
    response.raise_for_status()

    chunks = []
    for line in response.iter_lines():
        if not line:
            continue
        data = json.loads(line.decode("utf-8"))
        if "response" in data:
            chunks.append(data["response"])
        if data.get("done"):
            break

    return "".join(chunks).strip()


def get_run_id() -> str:
    """Return a simple timestamped run identifier for Stage 2 artifacts."""
    return datetime.utcnow().strftime("%Y%m%d_%H%M%S")

In [9]:
from dataclasses import dataclass
from typing import Any, Dict, List


@dataclass
class AgentIO:
    """Container for inputs and outputs passed between agents."""

    run_id: str
    payload: Dict[str, Any]


@dataclass
class AgentConfig:
    name: str
    system_prompt: str


class OllamaAgent:
    """Base class for simple JSON-in / JSON-out agents using Ollama.

    Each agent receives a JSON payload, formats a prompt, and expects **only JSON**
    in the response.
    """

    def __init__(self, config: AgentConfig):
        self.config = config

    def build_prompt(self, io: AgentIO) -> str:
        return (
            f"You are the {self.config.name} in a churn-retention system.\n"
            f"{self.config.system_prompt}\n\n"
            "You will receive JSON input (shown below). "
            "Use it only to reason; do not echo it back. "
            "Return ONLY valid JSON under the key 'output', no prose, no markdown, no code fences.\n\n"
            f"INPUT JSON:\n{json.dumps(io.payload, indent=2)}\n\n"
            "Respond with JSON only. Your response must start with '{' and end with '}'."
        )

    def __call__(self, io: AgentIO, *, temperature: float = 0.2, max_tokens: int = 1024) -> Dict[str, Any]:
        prompt = self.build_prompt(io)
        raw = call_ollama(prompt, temperature=temperature, max_tokens=max_tokens)

        start = raw.find("{")
        end = raw.rfind("}")
        if start == -1 or end == -1:
            raise ValueError(f"Agent {self.config.name} did not return JSON: {raw[:200]}")

        cleaned = raw[start : end + 1]

        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Agent {self.config.name} returned invalid JSON: {e}. Raw: {raw[:400]}"
            ) from e

        # Preferred contract: top-level {"output": {...}}
        if isinstance(parsed, dict) and "output" in parsed:
            return parsed["output"]

        # Fallback: sometimes the model returns the inner object directly
        if isinstance(parsed, dict) and "segments" in parsed:
            return parsed

        raise KeyError(
            f"Agent {self.config.name} response missing 'output' key. Top-level keys: {list(parsed.keys())}"
        )

## 2. Load Stage 1 data and prepare per-customer payload

Reads `high_risk_customers.csv`, keeps only the columns Stage 2 agents need (identifiers, churn metrics, enriched behavioral segments), and creates a timestamped run directory.

In [10]:
# Load Stage 1 high-risk customers and build a compact JSON payload

high_risk_df = pd.read_csv(HIGH_RISK_PATH)
print(f"Loaded {len(high_risk_df):,} high-risk customers")

stage2_cols = [
    "security_no",  # customer identifier
    "segment",
    "churn_probability",
    "churn_risk_score",
    "top_drivers",
    "last_purchase_days",
    "lifetime_value",
    "avg_order_value",
    # enriched behavioral features
    "value_segment",
    "engagement_segment",
    "price_sensitivity",
    "support_risk",
    "profile_tags",
]

missing = [c for c in stage2_cols if c not in high_risk_df.columns]
if missing:
    raise ValueError(f"Missing required columns for Stage 2: {missing}")

high_risk_stage2 = high_risk_df[stage2_cols].copy()

# For LLM-friendliness, we summarize per segment instead of sending all rows
segment_samples: Dict[str, List[Dict[str, Any]]] = {}
for seg, group in high_risk_stage2.groupby("segment"):
    # Take up to N example customers per segment
    samples = group.sample(n=min(100, len(group)), random_state=42).to_dict(orient="records")
    segment_samples[seg] = samples

run_id = get_run_id()
run_dir = STAGE2_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=True)

context_input = {
    "run_id": run_id,
    "segment_samples": segment_samples,
}

with open(run_dir / "context_input.json", "w") as f:
    json.dump(context_input, f, indent=2)

print(f"Prepared Stage 2 context input in {run_dir}")

Loaded 18,585 high-risk customers
Prepared Stage 2 context input in ../outputs/stage2/20260306_223043


## 3. Baseline strategy and agents

`compute_baseline_strategy` picks a default **intervention_type** and **channel** from behavioral segments (support_risk, price_sensitivity, engagement_segment, value_segment). The StrategyAgent refines this baseline; the ContentAgent writes email/SMS copy from the chosen strategy. 

In [11]:
# Deterministic baseline strategy from enriched features (used to constrain the LLM)

def compute_baseline_strategy(customer: Dict[str, Any]) -> Dict[str, str]:
    """Choose a default intervention_type and channel from behavioral segments.

    Priority: support_risk first, then price_sensitivity, engagement, value.
    """
    support_risk = int(customer.get("support_risk", 0))
    price_sensitivity = (customer.get("price_sensitivity") or "normal").lower()
    value_segment = (customer.get("value_segment") or "medium").lower()
    engagement_segment = (customer.get("engagement_segment") or "medium").lower()
    segment = (customer.get("segment") or "").lower()

    if support_risk == 1:
        # Service-recovery is implemented as a scalable check-in email (not 1:1 calls)
        return {"intervention_type": "service-recovery", "channel": "email"}

    if price_sensitivity == "high" and value_segment in ("low", "medium"):
        return {"intervention_type": "discount", "channel": "email"}

    if engagement_segment == "low":
        return {"intervention_type": "education", "channel": "email"}

    if value_segment == "high" or "vip" in segment:
        return {"intervention_type": "vip-benefit", "channel": "email"}

    if price_sensitivity == "high":
        return {"intervention_type": "discount", "channel": "email"}

    return {"intervention_type": "education", "channel": "email"}

## 4. Validation

Prints counts of `intervention_type` for the generated campaign plans to confirm a diverse strategy mix.

In [ ]:
# Strategy and content agents for per-customer campaign plans

strategy_agent = OllamaAgent(
    AgentConfig(
        name="StrategyAgent",
        system_prompt=(
            "You are a retention strategy expert. You will receive 'customer' and 'baseline_strategy'. "
            "The baseline_strategy was computed from behavioral rules; your job is to use it as the default "
            "and only change it if there is a strong reason (explain in reasoning).\n\n"
            "Allowed intervention_type values (pick exactly one): discount, education, vip-benefit, service-recovery.\n"
            "Allowed channel values: email, sms. Avoid phone calls; assume this is a large-scale business where 1:1 calls are rare.\n\n"
            "Guardrails:\n"
            "- If support_risk=0, do NOT use service-recovery unless the baseline is already service-recovery; "
            "prefer discount, education, or vip-benefit.\n"
            "- Use service-recovery mainly when support_risk=1, implemented as an email or SMS check-in from support (not a live call).\n"
            "- Across a batch, service-recovery should not dominate (aim for a mix of discount, education, vip-benefit, and service-recovery).\n"
            "- Default to the baseline_strategy's intervention_type and channel; if you override, your reasoning "
            "must cite the specific customer fields (value_segment, engagement_segment, price_sensitivity, support_risk, profile_tags) that justify it.\n\n"
            "Customer fields include: value_segment, engagement_segment, price_sensitivity, support_risk, profile_tags, "
            "segment, churn_probability, last_purchase_days, lifetime_value, avg_order_value, top_drivers.\n\n"
            "Respond ONLY with JSON of the form:\n"
            "{\n  'output': {\n    'security_no': str,\n    'segment': str,\n    'churn_probability': float,\n    'churn_risk_score': int,\n    'intervention_type': str,\n    'primary_goal': str,\n    'offer_type': str,\n    'channel': str,\n    'treatment_label': str,\n    'reasoning': str\n  }\n}\n\n"
            "Do not include any explanations, markdown, or extra keys outside this structure."
        ),
    )
)

content_agent = OllamaAgent(
    AgentConfig(
        name="ContentAgent",
        system_prompt=(
            "You are a retention copywriter. Given a customer profile and a chosen "
            "strategy (intervention_type, offer_type, channel, primary_goal), write practical, "
            "ready-to-send copy.\n\n"
            "Important: Do NOT mention 'churn' or 'churn risk' in the copy. Frame messages as proactive "
            "value, support, or offers (e.g. 'We want to make sure you get the most out of...', "
            "'Here\'s an exclusive offer...', 'We\'re here to help...'). Do NOT ask the customer to call us; "
            "instead, ask them to click a link, reply to the email/SMS, or visit their account.\n\n"
            "Use customer fields (value_segment, engagement_segment, profile_tags) to tailor tone. "
            "You will receive 'customer' and 'strategy'. Produce email_subject, email_body, and sms_body.\n\n"
            "Rules: Be friendly, clear, concise. Mention the offer or benefit explicitly. Include a clear "
            "call-to-action based on email/SMS (click or reply). Do not invent personal data. Keep SMS short "
            "for a single message.\n\n"
            "Respond ONLY with JSON of the form:\n"
            "{\n  'output': {\n    'security_no': str,\n    'channel': str,\n    'email_subject': str,\n    'email_body': str,\n    'sms_body': str\n  }\n}\n\n"
            "Do not include any explanations, markdown, or extra keys outside this structure."
        ),
    )
)


def generate_customer_campaign_plans(df: pd.DataFrame, max_customers: int = 20) -> List[Dict[str, Any]]:
    """Generate a JSON-structured campaign plan (strategy + content) per customer.

    This calls the StrategyAgent and ContentAgent once each per customer. It is
    computationally expensive but conceptually scales to large datasets if you
    are willing to wait. By default we limit to `max_customers=100` for faster
    experimentation in this notebook.
    """
    plans: List[Dict[str, Any]] = []

    df_iter = df.head(max_customers) if max_customers is not None else df

    for _, row in df_iter.iterrows():
        customer = row.to_dict()
        baseline_strategy = compute_baseline_strategy(customer)

        # 1) Strategy step (agent refines baseline; must return full strategy)
        strat_io = AgentIO(
            run_id=run_id,
            payload={"customer": customer, "baseline_strategy": baseline_strategy},
        )
        strategy = strategy_agent(strat_io, temperature=0.2, max_tokens=384)
        # Fallback to baseline if agent omitted intervention_type or channel
        if not (strategy.get("intervention_type") and strategy.get("channel")):
            strategy["intervention_type"] = strategy.get("intervention_type") or baseline_strategy["intervention_type"]
            strategy["channel"] = strategy.get("channel") or baseline_strategy["channel"]

        # 2) Content step (uses both customer and strategy)
        content_io = AgentIO(run_id=run_id, payload={"customer": customer, "strategy": strategy})
        message = content_agent(content_io, temperature=0.4, max_tokens=512)

        plan = {
            "security_no": str(customer.get("security_no")),
            "segment": customer.get("segment"),
            "churn_probability": float(customer.get("churn_probability", 0.0)),
            "churn_risk_score": int(customer.get("churn_risk_score", 0)),
            "strategy": strategy,
            "message": message,
        }

        plans.append(plan)

    return plans


# per-customer campaign plan generation on a sample of customers
customer_campaign_plans = generate_customer_campaign_plans(high_risk_stage2, max_customers=20)

with open(run_dir / "customer_campaign_plans.json", "w") as f:
    json.dump({"run_id": run_id, "customers": customer_campaign_plans}, f, indent=2)

print(f"Wrote campaign plans for {len(customer_campaign_plans)} customers to customer_campaign_plans.json")

Wrote campaign plans for 20 customers to customer_campaign_plans.json


In [13]:
strat_counts = pd.DataFrame([p.get("strategy", {}) for p in customer_campaign_plans])
print("intervention_type counts:")
print(strat_counts["intervention_type"].value_counts().to_string())
print()

intervention_type counts:
intervention_type
service-recovery    15
discount             2
education            2
vip-benefit          1

